# Taller 4 - Experimentación con MLflow (Covertype)

Flujo completo del taller:

1. **Datos crudos**: el CSV de Covertype se carga tal cual (todo `TEXT`) en la tabla `raw_covertype` de la base de datos de datos (`postgres-data`).
2. **Datos procesados**: se tipan, se limpian y se separan en train/test en la tabla `clean_covertype` de la misma base.
3. **Experimentación**: se entrenan 24 Random Forest con distintos hiperparámetros, leyendo los datos procesados desde la base de datos. Cada entrenamiento es un *run* anidado de MLflow con sus parámetros, métricas y modelo.
4. **Registro**: cada modelo queda registrado como una versión de `covertype_rf` en el Model Registry y el mejor recibe el alias `champion`, que es el que usa la API de inferencia.

La metadata de MLflow vive en su propio Postgres (`postgres-mlflow`) y los artefactos en MinIO (bucket `mlflows3`).

In [ ]:
import io
import os
import time
from datetime import datetime

import mlflow
import pandas as pd
import psycopg
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sqlalchemy import create_engine, inspect, text

In [ ]:
CSV_PATH = os.environ.get("CSV_PATH", "/home/jovyan/data/covertype.csv")
DATA_DB_URL = os.environ["DATA_DB_URL"]
PG_URL = DATA_DB_URL.replace("postgresql+psycopg://", "postgresql://")

EXPERIMENT_NAME = "covertype_rf"
MODEL_NAME = "covertype_rf"
CHAMPION_ALIAS = "champion"

SAMPLE_SIZE = 50_000
TEST_SIZE = 0.2
RANDOM_STATE = 42
FORCE_RELOAD = False

NUMERIC_FEATURES = [
    "elevation",
    "aspect",
    "slope",
    "horizontal_distance_to_hydrology",
    "vertical_distance_to_hydrology",
    "horizontal_distance_to_roadways",
    "hillshade_9am",
    "hillshade_noon",
    "hillshade_3pm",
    "horizontal_distance_to_fire_points",
]
CATEGORICAL_FEATURES = ["wilderness_area", "soil_type"]
TARGET = "cover_type"

engine = create_engine(DATA_DB_URL)

print("tracking uri:", mlflow.get_tracking_uri())
print("s3 endpoint :", os.environ.get("MLFLOW_S3_ENDPOINT_URL"))

## 1. Datos crudos en la base de datos

El CSV completo se copia con `COPY` de Postgres a la tabla `raw_covertype`. Las columnas quedan en `TEXT`, sin ningún procesamiento, solo con los nombres en minúscula.

Validación antes de cargar:

- Si `raw_covertype` ya existe y tiene el mismo número de filas que el CSV, **no se recarga**.
- Si no existe o está incompleta (por ejemplo, una carga interrumpida), se recrea y se carga de nuevo.
- Con `FORCE_RELOAD = True` se recarga siempre.

In [ ]:
def table_count(table: str) -> int | None:
    with engine.connect() as conn:
        if not inspect(conn).has_table(table):
            return None
        return conn.execute(text(f"SELECT count(*) FROM {table}")).scalar()


with open(CSV_PATH, "rb") as f:
    csv_rows = sum(1 for _ in f) - 1

raw_rows = table_count("raw_covertype")
raw_reloaded = FORCE_RELOAD or raw_rows != csv_rows

if not raw_reloaded:
    print(f"raw_covertype ya tiene las {raw_rows} filas del CSV, no se recarga")
else:
    raw_columns = [c.lower() for c in pd.read_csv(CSV_PATH, nrows=0).columns]
    with psycopg.connect(PG_URL) as conn, conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS raw_covertype")
        cur.execute(
            "CREATE TABLE raw_covertype ("
            "id SERIAL PRIMARY KEY, "
            + ", ".join(f"{c} TEXT" for c in raw_columns)
            + ", loaded_at TIMESTAMPTZ DEFAULT now())"
        )
        copy_sql = f"COPY raw_covertype ({', '.join(raw_columns)}) FROM STDIN WITH (FORMAT csv, HEADER true)"
        with cur.copy(copy_sql) as copy, open(CSV_PATH, "rb") as f:
            while data := f.read(1 << 20):
                copy.write(data)
    print(f"raw_covertype cargada: {table_count('raw_covertype')} filas (CSV: {csv_rows})")

## 2. Procesamiento y datos procesados en la base de datos

Desde `raw_covertype`:

- Variables numéricas a `float` (lo que no es número queda nulo).
- `cover_type` a entero y textos sin espacios sobrantes.
- Se eliminan filas incompletas y duplicadas.
- Split estratificado train/test (80/20) guardado en la columna `split`.

El resultado se guarda en `clean_covertype`. Solo se reprocesa si la tabla cruda se acaba de recargar o si `clean_covertype` no existe o está vacía. Si se cambia la lógica de procesamiento, hay que poner `FORCE_RELOAD = True` para regenerarla.

In [ ]:
def save_dataframe(df: pd.DataFrame, table: str) -> None:
    """Recrea la tabla con el esquema del DataFrame y copia las filas con COPY."""
    df.head(0).to_sql(table, engine, if_exists="replace", index=False)
    buffer = io.StringIO()
    df.to_csv(buffer, index=False, header=False)
    buffer.seek(0)
    with psycopg.connect(PG_URL) as conn, conn.cursor() as cur:
        with cur.copy(f"COPY {table} ({', '.join(df.columns)}) FROM STDIN WITH (FORMAT csv)") as copy:
            copy.write(buffer.getvalue())

In [ ]:
clean_rows = table_count("clean_covertype")

if not raw_reloaded and clean_rows:
    print(f"clean_covertype ya existe con {clean_rows} filas, no se reprocesa")
else:
    raw = pd.read_sql_table("raw_covertype", engine)

    clean = raw[NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET]].copy()
    for col in NUMERIC_FEATURES:
        clean[col] = pd.to_numeric(clean[col], errors="coerce").astype("float64")
    for col in CATEGORICAL_FEATURES:
        clean[col] = clean[col].str.strip()
    clean[TARGET] = pd.to_numeric(clean[TARGET], errors="coerce")

    filas_raw = len(clean)
    clean = clean.dropna().drop_duplicates().reset_index(drop=True)
    clean[TARGET] = clean[TARGET].astype("int64")

    train_idx, _ = train_test_split(
        clean.index, test_size=TEST_SIZE, stratify=clean[TARGET], random_state=RANDOM_STATE
    )
    clean["split"] = "test"
    clean.loc[train_idx, "split"] = "train"

    save_dataframe(clean, "clean_covertype")
    print(f"clean_covertype procesada: filas raw {filas_raw}, filas procesadas {len(clean)}")

In [ ]:
with engine.connect() as conn:
    print(pd.read_sql(text("SELECT split, count(*) AS filas FROM clean_covertype GROUP BY split"), conn))

## 3. Datos de entrenamiento desde la base de datos

El entrenamiento lee **solo** de `clean_covertype`. Para que las ejecuciones sean rápidas se toma una muestra de `SAMPLE_SIZE` filas de train y la cuarta parte de eso de test.

In [ ]:
data = pd.read_sql_table("clean_covertype", engine)

train_df = data[data["split"] == "train"].sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)
test_df = data[data["split"] == "test"].sample(n=SAMPLE_SIZE // 4, random_state=RANDOM_STATE)

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
X_train, y_train = train_df[FEATURES].astype({c: "object" for c in CATEGORICAL_FEATURES}), train_df[TARGET]
X_test, y_test = test_df[FEATURES].astype({c: "object" for c in CATEGORICAL_FEATURES}), test_df[TARGET]

print("train:", X_train.shape, "test:", X_test.shape)
y_train.value_counts().sort_index()

## 4. Pipeline

Igual que el ejemplo del profesor: `OneHotEncoder` para las variables categóricas (`handle_unknown="ignore"` para que un `soil_type` nuevo no rompa la predicción) y el resto pasa directo al `RandomForestClassifier`.

In [ ]:
def build_pipeline(**rf_params) -> Pipeline:
    column_trans = make_column_transformer(
        (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        remainder="passthrough",
    )
    return Pipeline(steps=[
        ("column_trans", column_trans),
        ("RandomForestClassifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **rf_params)),
    ])


build_pipeline()

## 5. Experimentación: 24 ejecuciones

Grilla de hiperparámetros: `n_estimators` × `max_depth` × `max_features` = 3 × 4 × 2 = **24 combinaciones**.

Todo queda bajo un *run* padre con un *run* anidado por combinación. En cada uno se registran los parámetros, las métricas (`accuracy`, `f1_macro`, `f1_weighted`, `fit_time_s`) y el modelo con su firma y un ejemplo de entrada. Cada modelo se registra como una nueva versión de `covertype_rf`.

In [ ]:
param_grid = list(ParameterGrid({
    "n_estimators": [25, 50, 100],
    "max_depth": [5, 10, 15, 20],
    "max_features": ["sqrt", "log2"],
}))
print("combinaciones:", len(param_grid))

In [ ]:
mlflow.set_experiment(EXPERIMENT_NAME)

results = []
parent_name = f"grid_rf_{datetime.now():%Y%m%d_%H%M%S}"

with mlflow.start_run(run_name=parent_name) as parent_run:
    mlflow.log_params({"sample_size": SAMPLE_SIZE, "n_combinations": len(param_grid), "random_state": RANDOM_STATE})
    mlflow.set_tags({"data_table": "clean_covertype", "model_name": MODEL_NAME})
    mlflow.log_input(
        mlflow.data.from_pandas(train_df, name="clean_covertype_train", targets=TARGET),
        context="training",
    )

    for i, params in enumerate(param_grid, start=1):
        with mlflow.start_run(run_name=f"rf_{i:02d}", nested=True) as run:
            pipe = build_pipeline(**params)
            inicio = time.perf_counter()
            pipe.fit(X_train, y_train)
            fit_time = time.perf_counter() - inicio

            y_pred = pipe.predict(X_test)
            metrics = {
                "accuracy": accuracy_score(y_test, y_pred),
                "f1_macro": f1_score(y_test, y_pred, average="macro"),
                "f1_weighted": f1_score(y_test, y_pred, average="weighted"),
                "fit_time_s": fit_time,
            }

            mlflow.log_params(params)
            mlflow.log_metrics(metrics)
            model_info = mlflow.sklearn.log_model(
                sk_model=pipe,
                name="model",
                signature=infer_signature(X_test, y_pred),
                input_example=X_test.head(3),
                registered_model_name=MODEL_NAME,
                skops_trusted_types=["sklearn.tree._tree.Tree"],
            )

            results.append({
                "run": f"rf_{i:02d}",
                "run_id": run.info.run_id,
                "version": model_info.registered_model_version,
                **params,
                **metrics,
            })
            print(f"rf_{i:02d} {params} -> accuracy={metrics['accuracy']:.4f} f1_macro={metrics['f1_macro']:.4f}")

## 6. Resultados y modelo campeón

Se escoge el mejor por `f1_macro` (las clases están desbalanceadas, así que pesa igual acertar en las clases pequeñas) y se le asigna el alias `champion` en el Model Registry. La API carga siempre `models:/covertype_rf@champion`, así que mover el alias cambia el modelo que sirve sin reiniciarla.

In [ ]:
results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
results_df

In [ ]:
client = MlflowClient()
best = results_df.iloc[0]

client.set_registered_model_alias(MODEL_NAME, CHAMPION_ALIAS, str(best["version"]))
client.update_registered_model(
    MODEL_NAME,
    description="Random Forest para predecir el tipo de cobertura forestal (Covertype). El alias 'champion' apunta al mejor f1_macro.",
)
client.update_model_version(
    MODEL_NAME,
    str(best["version"]),
    description=f"Mejor del grid {parent_name}: accuracy={best['accuracy']:.4f}, f1_macro={best['f1_macro']:.4f}",
)

print(f"champion -> {MODEL_NAME} v{best['version']} (run {best['run']})")
best

## 7. Verificación: cargar el campeón desde MLflow

Se carga el modelo exactamente como lo hace la API (`models:/<nombre>@champion`) y se predice sobre unas filas de test.

In [ ]:
champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@{CHAMPION_ALIAS}")

muestra = X_test.head(5)
pd.DataFrame({"real": y_test.head(5).to_numpy(), "prediccion": champion.predict(muestra)})